# Chapter 4: Data Manipulation in R

## 1. Introduction

Welcome to data subsetting and indexing, one of the most fundamental and frequently used skills in data analysis. In this section, you will learn how to select, filter, and remove data from R objects by position, name, or logical conditions. This skill is paramount in precision health, where you must navigate vast datasets to find the specific information you need. Our learning journey will begin with simple vectors, progress to two-dimensional matrices, and culminate in the powerful and flexible data frame, preparing you to isolate specific patient cohorts, analyze subgroups, or prepare data for modeling.

---

## 2. Key Concepts and Definitions

*   **Indexing**: The process of accessing data elements within an object using their position or name. In a clinical trial dataset, this could mean retrieving the record for the 5th patient (`data[5, ]`) or accessing the 'blood_pressure' column (`data$sbp_mmhg`). R uses 1-based indexing, meaning the first element is at position 1.
*   **Subsetting**: The action of creating a smaller dataset from a larger one based on specific criteria. This is the primary tool for cohort selection, such as creating a new data frame containing only patients over the age of 65 with a history of hypertension.
*   **Logical Filtering**: A type of subsetting where data is selected if it meets a `TRUE` condition. For example, `patients$age > 50` creates a logical vector of `TRUE`/`FALSE` values that can be used to filter a `patients` data frame, effectively selecting the cohort of patients older than 50.
*   **`drop = FALSE`**: An argument used in matrix and data frame subsetting to prevent R from simplifying the data structure. For instance, when selecting a single column, using `drop = FALSE` ensures the output is a one-column data frame, not a vector. This maintains structural consistency, which is critical for preventing errors in automated analysis scripts.

---

## 3. Main Content

### 3.1 Subsetting Vectors

**By Position**

Select vector elements using their numerical index. Remember, R uses 1-based indexing.

```R
# Vector of patient hematocrit levels (%)
hematocrit_levels <- c(45, 38, 51, 42)

# Select the third patient's level
hematocrit_levels[3]
# [1] 51
```

**By Logical Condition**

Select elements that meet a specific condition. This is a powerful way to filter data based on clinical thresholds.

> **Medical Background:** Polycythemia is a condition characterized by an abnormally high concentration of red blood cells, leading to elevated hematocrit levels. While diagnostic criteria are complex, a simplified threshold like >50% is often used in initial data screening to flag patients for further review.

```R
# Select levels indicating potential polycythemia (>50%)
hematocrit_levels[hematocrit_levels > 50]
# [1] 51
```

### 3.2 Subsetting Matrices

Matrices use `[row, column]` notation. Leaving a dimension blank selects all elements in that dimension.

> **Important:** Use `drop = FALSE` to ensure the output of a subset operation remains a matrix or data frame. R's default behavior of simplifying single-column or single-row selections into a vector can break scripts that expect a two-dimensional structure, leading to unpredictable errors in analysis pipelines.

```R
# Matrix of patient vitals (Row 1: Patient A, Row 2: Patient B)
vitals_matrix <- matrix(c(45, 140, 72, 62, 165, 85), nrow = 2, byrow = TRUE)
colnames(vitals_matrix) <- c("age", "sbp_mmhg", "hr_bpm")

# Get systolic BP, which R simplifies to a vector by default
bp_vector <- vitals_matrix[, "sbp_mmhg"] 
print(bp_vector)
# sbp_mmhg sbp_mmhg 
#      140      165 

# Get systolic BP as a single-column matrix using drop = FALSE
bp_matrix <- vitals_matrix[, "sbp_mmhg", drop = FALSE]
print(bp_matrix)
#      sbp_mmhg
# [1,]      140
# [2,]      165
```

### 3.3 Subsetting Data Frames

**Create a Sample Data Frame**

```R
patients <- data.frame(
  patient_id = c("PT000001", "PT000002", "PT000003", "PT000004"),
  age = c(45, 62, 38, 55),
  sbp_mmhg = c(140, 165, 120, 135),
  treatment_group = c("A", "B", "A", "B")
)
```

**Select by Position**

Use `[row, column]` notation. For example, `patients[2, ]` selects all columns for the second row. To select multiple non-consecutive rows or columns, use `c()`, as in `patients[c(1, 3), c("age", "sbp_mmhg")]`, which selects specified columns for rows 1 and 3.

```R
# Get all data for the second patient
patients[2, ]
#   patient_id age sbp_mmhg treatment_group
# 2   PT000002  62      165               B

# Get the first and third patients' age and SBP
patients[c(1, 3), c("age", "sbp_mmhg")]
#   age sbp_mmhg
# 1  45      140
# 3  38      120
```

**Select Columns by Name**

> **Pro Tip: Choosing the Right Column Selector**
> - Use `$` for quick, interactive analysis where column names are fixed.
> - Use `[[ ]]` when the column name is stored in a variable, which is essential for programming and creating reusable functions.
> - Use `[ ]` to select one or more columns while guaranteeing the output is a data frame, which is the safest option for scripting.

```R
# Get age as a vector for calculation
mean(patients$age)
# [1] 50

# Get age as a data frame
age_df <- patients["age"]
print(age_df)
#   age
# 1  45
# 2  62
# 3  38
# 4  55

# Use a variable name to select a column (safer in scripts)
col_name <- "sbp_mmhg"
patients[[col_name]]
# [1] 140 165 120 135
```

**Drop Columns**

```R
# Method 1: Modify in-place by assigning NULL
patients_copy <- patients
patients_copy$treatment_group <- NULL
print(patients_copy)
#   patient_id age sbp_mmhg
# 1   PT000001  45      140
# 2   PT000002  62      165
# 3   PT000003  38      120
# 4   PT000004  55      135

# Method 2: Create a new data frame without the column
patients_simplified <- patients[, !names(patients) %in% c("treatment_group")]
print(patients_simplified)
#   patient_id age sbp_mmhg
# 1   PT000001  45      140
# 2   PT000002  62      165
# 3   PT000003  38      120
# 4   PT000004  55      135
```

**Logical Filtering**

> **Pro Tip:** Use `&` for AND and `|` for OR when combining logical tests. For debugging complex filters, the `which()` function is invaluable. It returns the numerical indices of the rows that meet your condition, allowing you to inspect exactly which data points are being selected.

```R
# Select all patients older than 60
patients[patients$age > 60, ]
#   patient_id age sbp_mmhg treatment_group
# 2   PT000002  62      165               B

# Select patients in group "B" AND with SBP > 150
patients[patients$treatment_group == "B" & patients$sbp_mmhg > 150, ]
#   patient_id age sbp_mmhg treatment_group
# 2   PT000002  62      165               B

# Use which() to get row numbers for debugging
high_risk_indices <- which(patients$age > 60) # Returns 2
patients[high_risk_indices, ]
#   patient_id age sbp_mmhg treatment_group
# 2   PT000002  62      165               B
```

> **In Practice:** At City Medical Center, clinical researchers constantly filter large electronic health record (EHR) datasets. A query like `patients[patients$age < 40 | patients$treatment_group == "B", ]` could be used to identify all participants for a follow-up study that targets either a younger demographic or anyone who received a specific new treatment, regardless of age. This is a fundamental step in clinical data science.

```R
# Select patients younger than 40 OR in treatment group "B"
patients[patients$age < 40 | patients$treatment_group == "B", ]
#   patient_id age sbp_mmhg treatment_group
# 2   PT000002  62      165               B
# 3   PT000003  38      120               A
# 4   PT000004  55      135               B
```

**Using the `subset()` Function**

As your filtering logic becomes more complex (e.g., involving three or more conditions), the `subset()` function can make your code significantly cleaner and easier for colleagues to understand.

```R
# Select patients older than 60 using subset()
subset(patients, age > 60)
#   patient_id age sbp_mmhg treatment_group
# 2   PT000002  62      165               B

# Select specific columns for patients in group "B" with SBP > 150
subset(patients, treatment_group == "B" & sbp_mmhg > 150, select = c(patient_id, sbp_mmhg))
#   patient_id sbp_mmhg
# 2   PT000002      165
```

---

## 4. Practice Exercises

### Exercise 1: Basic Column and Row Selection

**Objective:** Practice basic indexing to retrieve specific data points.
**Time:** 3 minutes
**Medical Context:** A clinician needs to quickly look up the age of a specific patient and the treatment group for another from a patient roster.

Using the `patients` data frame, perform the following two tasks:
1.  Select only the `age` of the third patient (`PT000003`).
2.  Select the entire row of data for the fourth patient (`PT000004`).

**📝 Your Solution:**

In [ ]:
# TODO: Write your solution here
# Hint: Check the task description above




<details>
<summary style="background-color: #f0f0f0; padding: 10px; cursor: pointer; border-radius: 5px;">
<strong>🔍 Click to reveal solution</strong>
</summary>

<div style="padding: 10px; border-left: 3px solid #2196F3; margin-top: 10px;">

**Solution**


```R
# 1. Select the age of the third patient
age_p03 <- patients[3, "age"]
print(age_p03)
# [1] 38

# 2. Select the entire row for the fourth patient
row_p04 <- patients[4, ]
print(row_p04)
#   patient_id age sbp_mmhg treatment_group
# 4   PT000004  55      135               B
```

**Explanation:** The first task uses `[row, column]` indexing to pinpoint a single value. The second task uses `[row, ]` with a blank column argument to select all columns for the specified row.
**Key Learning:** Basic `[row, column]` indexing is the foundation for all data selection tasks.

</div>
</details>

### Exercise 2: Create a High-Risk Cohort

**Objective:** Use logical filtering with multiple conditions to create a new data frame representing a specific patient cohort.
**Time:** 5 minutes
**Medical Context:** Dr. Chen is investigating risk factors for hypertensive events and has asked you to isolate a specific patient cohort from the study data.

Using the `patients` data frame, create a new data frame called `high_risk_cohort`. It should contain only the `patient_id` and `sbp_mmhg` for patients who are older than 50 **and** are in treatment group "B".

**📝 Your Solution:**

In [ ]:
# TODO: Write your solution here
# Hint: Check the task description above




<details>
<summary style="background-color: #f0f0f0; padding: 10px; cursor: pointer; border-radius: 5px;">
<strong>🔍 Click to reveal solution</strong>
</summary>

<div style="padding: 10px; border-left: 3px solid #2196F3; margin-top: 10px;">

**Solution**


```R
high_risk_cohort <- patients[
  patients$age > 50 & patients$treatment_group == "B",
  c("patient_id", "sbp_mmhg")
]

# Print the result to verify
print(high_risk_cohort)
#   patient_id sbp_mmhg
# 2   PT000002      165
# 4   PT000004      135
```

**Explanation:** The code first creates a logical vector by combining two conditions (`age > 50` and `treatment_group == "B"`) with the `&` (AND) operator. This logical vector is used to filter the rows. Then, a character vector `c("patient_id", "sbp_mmhg")` is used to select the desired columns.
**Key Learning:** Combining logical conditions with `&` (AND) or `|` (OR) is essential for defining precise patient cohorts.

> **Reflection Moment:** The `high_risk_cohort` data frame now contains a very specific subset of your original data. How might creating such isolated cohorts be useful in a real clinical trial? Consider aspects like targeted statistical analysis, generating reports for specific subgroups, or preparing data for visualization.

</div>
</details>

### Exercise 3: Advanced Filtering with `subset()`

**Objective:** Use the `subset()` function to perform complex filtering and column selection in a single, readable command.
**Time:** 5 minutes
**Medical Context:** For a presentation, you need to quickly generate a summary table of patients who are either younger than 40 or have a systolic blood pressure of 140 mmHg or higher. You only need to show their ID and treatment group.

Using the `patients` data frame, create a new data frame called `presentation_data` that meets these criteria. Use the `subset()` function.

**📝 Your Solution:**

In [ ]:
# TODO: Write your solution here
# Hint: Check the task description above




<details>
<summary style="background-color: #f0f0f0; padding: 10px; cursor: pointer; border-radius: 5px;">
<strong>🔍 Click to reveal solution</strong>
</summary>

<div style="padding: 10px; border-left: 3px solid #2196F3; margin-top: 10px;">

**Solution**


```R
presentation_data <- subset(
  patients,
  age < 40 | sbp_mmhg >= 140,
  select = c(patient_id, treatment_group)
)

# Print the result to verify
print(presentation_data)
#   patient_id treatment_group
# 1   PT000001               A
# 2   PT000002               B
# 3   PT000003               A
```

**Explanation:** The `subset()` function simplifies the syntax. The first argument is the data frame, the second is the logical condition (using `|` for OR), and the `select` argument takes a vector of column names to keep. This is often more readable than standard bracket notation for complex queries.
**Key Learning:** The `subset()` function is a powerful tool for improving code clarity when filtering and selecting data simultaneously.

</div>
</details>

---

## 5. Practical Applications

*   **Clinical Trial Cohort Selection:** Researchers use logical filtering to screen thousands of electronic health records (EHRs) to find eligible participants. For example, they might subset a dataset for patients aged 50-75, with a specific diagnosis code, and without certain comorbidities (`age >= 50 & age <= 75 & diagnosis == "I10"`). This ensures the study population is well-defined and meets regulatory standards.
*   **Pharmacovigilance and Drug Safety:** After a drug is on the market, safety analysts monitor adverse event databases. They use subsetting to isolate all reports for a specific drug (`drug_name == "DrugX"`) and a particular side effect (`adverse_event == "rash"`). This allows them to calculate reporting rates and detect potential safety signals that warrant further investigation.
*   **Genomic Data Analysis:** In precision oncology, analysts work with vast matrices of gene expression data where rows are genes and columns are patient samples. They use indexing to extract data for a specific panel of cancer-related genes (`genes_of_interest <- c("BRCA1", "TP53")`) and subset the columns to compare expression levels between tumor samples and healthy tissue samples (`sample_type == "Tumor"`). This is critical for identifying biomarkers and guiding personalized treatment.

---

## 6. Summary and Key Takeaways

In this section, we've explored the essential techniques for selecting and filtering data in R, a cornerstone of preparing clinical data for analysis. You learned how to precisely access and manipulate data in vectors, matrices, and data frames to isolate the information that matters most.

*   **Indexing with `[ ]`:** This is the fundamental operator for subsetting by position, name, or logical vectors.
*   **Column Selection Strategy:** Use `$` for interactive work, `[[ ]]` for programmatic access with variables, and `[ ]` to guarantee the result is a data frame.
*   **Logical Filtering:** Combine conditions with `&` (AND) and `|` (OR) to define and extract highly specific patient cohorts from large datasets.
*   **`subset()` Function:** This is a readable alternative for performing row filtering and column selection in one clean step, improving code maintainability.
*   **Data Integrity:** Using `drop = FALSE` is a key defensive programming technique to prevent unintended data type conversions and ensure your scripts run reliably.

Mastering these skills allows you to move from raw, complete datasets to the clean, focused subsets needed to answer specific clinical and research questions. In the next section, we will build upon this foundation to learn about data transformation, where we will modify existing variables and create new ones.

---